# Trained Machine Learning (ML) Models
We are going to take our previous files and set up actual ML testing. First, we must import our data we set up in the previous notebook. Because we altered the paths, we are going to perform some wizardry pathing to find it properly

## Import data
We have things left over in the artifacts folder from the last notebook, then transforms the data into an array we can teach our ML systems with

In [0]:
import os
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import issparse


nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_ws = nb_path.split("/notebooks/")[0]
repo_fs = "/Workspace" + repo_ws if not repo_ws.startswith("/Workspace") else repo_ws

artifacts_dir = os.path.join(repo_fs, "artifacts")

pipeline = joblib.load(os.path.join(artifacts_dir, "stedi_feature_pipeline.pkl"))

X_train = np.load(
    os.path.join(artifacts_dir, "X_train_transformed.npy"),
    allow_pickle=True
)

X_test = np.load(
    os.path.join(artifacts_dir, "X_test_transformed.npy"),
    allow_pickle=True
)

def to_float_matrix(arr: np.ndarray) -> np.ndarray:

   """

   Ensures that input arrays (possibly object-dtype, sparse, or 0-d) are converted to a 2-D float matrix.

   This is necessary because saved feature arrays may have inconsistent shapes or types after transformation,

   and ML models require numeric 2-D arrays for training and prediction.

   """

   if arr.ndim == 0:
       # Handle 0-d array directly
       arr = arr.item()
       if issparse(arr):
           arr = arr.toarray()
       arr = np.array(arr, dtype=float)
   elif arr.dtype == object:
       arr = np.array([
           x.toarray() if issparse(x) else np.array(x, dtype=float)
           for x in arr
       ])
       arr = np.vstack(arr)
   elif issparse(arr):
       arr = arr.toarray()
   else:
       arr = np.array(arr, dtype=float)
   return arr


X_train = to_float_matrix(X_train)
X_test = to_float_matrix(X_test)


y_train = pd.read_pickle(os.path.join(artifacts_dir, "y_train.pkl"))
y_test  = pd.read_pickle(os.path.join(artifacts_dir, "y_test.pkl"))


y_train = np.ravel(y_train)
y_test = np.ravel(y_test)


X_train.shape, X_test.shape, y_train.shape, y_test.shape


## Train Logistic Regression (Baseline Model)
Using the arrays we now have, we are going to train the Logistic Regression method of Machine Learning, imported from sklearn. The target goals for accuracy score is between 0.75 to 0.90; anything more is gold.

In [0]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=300)
log_reg.fit(X_train, y_train)

log_reg_score = log_reg.score(X_test, y_test)
log_reg_score


## Train Random Forest (Baseline Model)
Now we are going to run the arrays through a Random Forest model, to directly compare against the Logistic Regression. The goal for the value is between 0.85 and 0.95; anything more is gold.

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

rf_score = rf.score(X_test, y_test)
rf_score

Now we create a simple comparison dictionary.

In [0]:
results = {
    "Logistic Regression baseline": log_reg_score,
    "Random Forest baseline": rf_score
}
results

##Baseline Model Analysis
The Logistic Regression performed better by a 0.3% difference. I however think that Random Forest is more stable for noisy data; it took longer and the slightly lower score feels more accurate.
Why didn't Random Forest have a higher score? It took longer and has a higher baseline threshold requirement for accuracy; in comparison, its lower score for accuracy feels poor. Testing the model helps us know whether or not the ML is fit for work function and will give us the accuracy we desire before pushing it into a production environment.
If a model is wrong, anyone using the service could be impacted with improper predictions; the company using it could be held liable depending on the impact, and the trust relationship is lost between the customer and the producer.
Fairness is required to prevent improper decision making in all aspects of our lives; if we treat others fairly, we will be given and will find ourselves giving people the proper attention and time they deserve. If we are unfair with our fellow men, we may find ourselves afoul of people who desire to be unfair with us.

Upload the baseline models

In [0]:
import os
import joblib
from datetime import datetime

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_ws = nb_path.split("/notebooks/")[0]
repo_fs = "/Workspace" + repo_ws if not repo_ws.startswith("/Workspace") else repo_ws

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

artifacts_root = os.path.join(repo_fs, "artifacts")
base_dir = os.path.join(artifacts_root, "stedi_models", run_id)
os.makedirs(base_dir, exist_ok=True)

joblib.dump(log_reg, os.path.join(base_dir, "log_reg.joblib"))
joblib.dump(rf, os.path.join(base_dir, "random_forest.joblib"))

metadata = {
    "run_id": run_id,
    "logistic_regression_accuracy": float(log_reg_score),
    "random_forest_accuracy": float(rf_score),
}
joblib.dump(metadata, os.path.join(base_dir, "metadata.joblib"))

base_dir


Zip the baseline models

In [0]:
import shutil
import os

zip_base = os.path.join(os.path.dirname(base_dir), f"stedi_models_{run_id}")
zip_path = shutil.make_archive(zip_base, "zip", base_dir)

zip_path
